In [1]:
import os
import pandas as pd
import numpy as np

data_path = "../data/output"
output_path = "../data/output"

corpus = pd.read_csv(os.path.join(data_path, "CORPUS.csv"), sep="|")
corpus.head()

,book_number,chapter,verse,token_id,token_str,term_str,pos,pos_group
0,1,1,1,1,in,in,IN,OTHER
1,1,1,1,2,the,the,DT,OTHER
2,1,1,1,3,beginning,beginning,VBG,VERB
3,1,1,1,4,god,god,NN,NOUN
4,1,1,1,5,created,created,VBN,VERB


In [2]:
N = corpus.shape[0]

vocab = (
    corpus
    .groupby("term_str")
    .agg(
        n=("term_str","count")
    )
    .reset_index()
)

vocab["p"] = vocab["n"] / N
vocab["i"] = -np.log2(vocab["p"])
vocab.head()

,term_str,n,p,i
0,a,8177,0.010355,6.593531
1,aaron,319,0.000404,11.273474
2,aaron's,31,0.000039,14.636690
3,aaronites,2,0.000003,18.590887
4,abaddon,1,0.000001,19.590887


In [3]:
doc_count = corpus.groupby(["book_number","chapter","verse"]).ngroups

df = (
    corpus
    .groupby("term_str")
    .apply(lambda x: x[["book_number","chapter","verse"]].drop_duplicates().shape[0])
    .reset_index(name="df")
)

vocab = vocab.merge(df, on="term_str")

vocab["dfidf"] = vocab["df"] * vocab["i"]
vocab.head()

/var/folders/jd/kgmmf40n35d3bhgwcsrqn78r0000gn/T/ipykernel_15836/2259588357.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x[["book_number","chapter","verse"]].drop_duplicates().shape[0])


,term_str,n,p,i,df,dfidf
0,a,8177,0.010355,6.593531,6216,40985.387317
1,aaron,319,0.000404,11.273474,303,3415.862649
2,aaron's,31,0.000039,14.636690,31,453.737402
3,aaronites,2,0.000003,18.590887,2,37.181773
4,abaddon,1,0.000001,19.590887,1,19.590887


In [4]:
pos_mode = (
    corpus
    .groupby("term_str")["pos"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index(name="max_pos")
)

pos_group_mode = (
    corpus
    .groupby("term_str")["pos_group"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index(name="max_pos_group")
)

vocab = vocab.merge(pos_mode, on="term_str")
vocab = vocab.merge(pos_group_mode, on="term_str")

vocab.head()

,term_str,n,p,i,df,dfidf,max_pos,max_pos_group
0,a,8177,0.010355,6.593531,6216,40985.387317,DT,OTHER
1,aaron,319,0.000404,11.273474,303,3415.862649,NN,NOUN
2,aaron's,31,0.000039,14.636690,31,453.737402,NN,NOUN
3,aaronites,2,0.000003,18.590887,2,37.181773,NNS,NOUN
4,abaddon,1,0.000001,19.590887,1,19.590887,NN,NOUN


In [5]:
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

vocab["stop"] = vocab["term_str"].isin(stop_words)
vocab.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nicholasthornton/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,term_str,n,p,i,df,dfidf,max_pos,max_pos_group,stop
0,a,8177,0.010355,6.593531,6216,40985.387317,DT,OTHER,True
1,aaron,319,0.000404,11.273474,303,3415.862649,NN,NOUN,False
2,aaron's,31,0.000039,14.636690,31,453.737402,NN,NOUN,False
3,aaronites,2,0.000003,18.590887,2,37.181773,NNS,NOUN,False
4,abaddon,1,0.000001,19.590887,1,19.590887,NN,NOUN,False


In [6]:
vocab = vocab[[
    "term_str",
    "n",
    "p",
    "i",
    "df",
    "dfidf",
    "max_pos",
    "max_pos_group",
    "stop"
]]

vocab.shape

(12878, 9)

In [9]:
top20 = (
    vocab[~vocab["stop"]]
    .sort_values("dfidf", ascending=False)
    .head(20)
)

top20[["term_str","dfidf"]]

,term_str,dfidf
12010,unto,47513.645791
6929,lord,44376.152770
10119,shall,38352.681897
4845,god,28971.248761
11550,thou,27836.499604
9739,said,27466.918221
11601,thy,22589.614820
12684,ye,21742.454603
11475,thee,21036.798984
7119,man,19197.791035


In [11]:
delimiter = "|"
print("Delimiter:", delimiter)

num_terms = vocab.shape[0]
print("Number of observations:", num_terms)

columns_string = "|".join(vocab.columns)
print("Columns:", columns_string)

top20 = (
    vocab[~vocab["stop"]]
    .sort_values("dfidf", ascending=False)
    .head(20)
)

print("Top 20 significant words by DFIDF (excluding stopwords):")
for word in top20["term_str"]:
    print("-", word)

Delimiter: |
Number of observations: 12878
Columns: term_str|n|p|i|df|dfidf|max_pos|max_pos_group|stop
Top 20 significant words by DFIDF (excluding stopwords):
- unto
- lord
- shall
- god
- thou
- said
- thy
- ye
- thee
- man
- israel
- upon
- came
- people
- come
- hath
- also
- son
- king
- house


In [12]:
vocab.to_csv(os.path.join(output_path, "VOCAB.csv"), index=False, sep="|")
print("VOCAB saved.")
print("Shape:", vocab.shape)

VOCAB saved.
Shape: (12878, 9)
